# 08 — Consultas Generales

Este notebook contiene 30 consultas de ejemplo utilizando SQL dentro de PySpark para explorar el modelo dimensional (SIAF) y convirtiendo los resultados a Pandas para una visualización tabulada interactiva.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from src.paths import GOLD, str_path
from src.spark_utils import get_spark

# Configurar el display máximo de filas en pandas para notebooks
pd.set_option('display.max_rows', 100)

# Inicializamos Spark
spark = get_spark(app_name="MEF_Consultas", memory="2g")
print("Spark inicializado correctamente.")

[OK] SparkSession lista | version=3.5.0 | master=local[*]
Spark inicializado correctamente.


In [2]:
# Cargar dimensiones y tabla de hechos
dim_muni = spark.read.parquet(str_path(GOLD["dim_municipalidad"]))
dim_geo = spark.read.parquet(str_path(GOLD["dim_geografia"]))
dim_tiempo = spark.read.parquet(str_path(GOLD["dim_tiempo"]))
dim_clas = spark.read.parquet(str_path(GOLD["dim_clasificacion_ingreso"]))
dim_fin = spark.read.parquet(str_path(GOLD["dim_financiamiento"]))
fact_ejec = spark.read.parquet(str_path(GOLD["fact_ejecucion_presupuestal"]))

# Registrar como vistas temporales para usar Spark SQL
dim_muni.createOrReplaceTempView("dim_municipalidad")
dim_geo.createOrReplaceTempView("dim_geografia")
dim_tiempo.createOrReplaceTempView("dim_tiempo")
dim_clas.createOrReplaceTempView("dim_clasificacion")
dim_fin.createOrReplaceTempView("dim_financiamiento")
fact_ejec.createOrReplaceTempView("fact_ejec")

print("Vistas temporales creadas correctamente para todas las tablas del modelo dimensional.")

Vistas temporales creadas correctamente para todas las tablas del modelo dimensional.


## 1. ¿Cuántas municipalidades hay registradas?

In [3]:
query_1 = """
SELECT COUNT(*) as Total_Municipalidades
FROM dim_municipalidad
"""
spark.sql(query_1).toPandas()

,Total_Municipalidades
0,1964


## 2. ¿Cuántas mancomunidades hay registradas?

In [4]:
query_2 = """
SELECT COUNT(*) as Total_Mancomunidades
FROM dim_municipalidad
WHERE LOWER(Ejecutora) LIKE '%mancomunidad%'
"""
spark.sql(query_2).toPandas()

,Total_Mancomunidades
0,70


## 3. ¿Cuántas municipalidades tienen el nombre "Santa Rosa"?

In [9]:
query_3 = """
SELECT DISTINCT 
    m.Ejecutora as Nombre_Municipalidad, 
    m.Categoria_Municipal, 
    g.UBIGEO
FROM dim_municipalidad m
JOIN fact_ejec f ON m.SK_Municipalidad = f.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE LOWER(m.Ejecutora) LIKE '%santa rosa%'
"""
spark.sql(query_3).toPandas()

,Nombre_Municipalidad,Categoria_Municipal,UBIGEO
0,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,021510
1,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,210504
2,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,F,210808
3,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,F,030710
4,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,220304
5,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,C,150139
6,MUNICIPALIDAD DISTRITAL DE SANTA ROSA DE ALTO ...,F,100705
7,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,010610
8,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,E,050507
9,MUNICIPALIDAD DISTRITAL DE SANTA ROSA DE SACCO,D,120808


## 4. ¿Cuántas municipalidades hay en Lima Metropolitana?

In [6]:
query_4 = """
SELECT COUNT(DISTINCT m.SK_Municipalidad) as Total_Muni_Lima_Metropolitana
FROM dim_municipalidad m
JOIN fact_ejec f ON m.SK_Municipalidad = f.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Departamento = 'LIMA' AND g.Nombre_Provincia = 'LIMA'
"""
spark.sql(query_4).toPandas()

,Total_Muni_Lima_Metropolitana
0,49


## 5. Detalle de municipalidades en Lima Metropolitana

In [8]:
query_5 = """
SELECT DISTINCT m.Ejecutora as Municipalidad, g.Nombre_Distrito as Distrito
FROM dim_municipalidad m
JOIN fact_ejec f ON m.SK_Municipalidad = f.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Departamento = 'LIMA' AND g.Nombre_Provincia = 'LIMA'
ORDER BY Distrito
"""
spark.sql(query_5).toPandas()

,Municipalidad,Distrito
0,MUNICIPALIDAD DISTRITAL DE ANCON,ANCON
1,MUNICIPALIDAD DISTRITAL DE ATE - VITARTE,ATE
2,MUNICIPALIDAD DISTRITAL DE BARRANCO,BARRANCO
3,MUNICIPALIDAD DISTRITAL DE BREÃA,BRENA
4,MUNICIPALIDAD DISTRITAL DE CARABAYLLO,CARABAYLLO
5,MUNICIPALIDAD DISTRITAL DE CHACLACAYO,CHACLACAYO
6,MUNICIPALIDAD DISTRITAL DE CHORRILLOS,CHORRILLOS
7,MUNICIPALIDAD DISTRITAL DE CIENEGUILLA,CIENEGUILLA
8,MUNICIPALIDAD DISTRITAL DE COMAS,COMAS
9,MANCOMUNIDAD MUNICIPAL LIMA NORTE,COMAS


## 6. Distribución de municipalidades por Categoría Municipal

In [10]:
query_6 = """
SELECT Categoria_Municipal, COUNT(*) as Cantidad
FROM dim_municipalidad
GROUP BY Categoria_Municipal
ORDER BY Cantidad DESC
"""
spark.sql(query_6).toPandas()

,Categoria_Municipal,Cantidad
0,G,642
1,F,523
2,E,388
3,D,138
4,B,130
5,A,93
6,C,48
7,SIN_CATEGORIA,2


## 7. Distribución de municipalidades por Nivel de Gobierno

In [11]:
query_7 = """
SELECT Nivel_Gobierno, COUNT(*) as Cantidad
FROM dim_municipalidad
GROUP BY Nivel_Gobierno
ORDER BY Cantidad DESC
"""
spark.sql(query_7).toPandas()

,Nivel_Gobierno,Cantidad
0,GOBIERNOS LOCALES,1964


## 8. Total de distritos registrados en la dimensión geográfica

In [12]:
query_8 = """
SELECT COUNT(DISTINCT Cod_Departamento, Cod_Provincia, Cod_Distrito) as Total_Distritos
FROM dim_geografia
"""
spark.sql(query_8).toPandas()

,Total_Distritos
0,1896


## 9. Top 10 Departamentos con mayor cantidad de distritos

In [13]:
query_9 = """
SELECT Nombre_Departamento, COUNT(DISTINCT Cod_Distrito) as Total_Distritos
FROM dim_geografia
GROUP BY Nombre_Departamento
ORDER BY Total_Distritos DESC
LIMIT 10
"""
spark.sql(query_9).toPandas()

,Nombre_Departamento,Total_Distritos
0,LIMA,43
1,JUNIN,36
2,AREQUIPA,29
3,HUANCAVELICA,25
4,AMAZONAS,23
5,AYACUCHO,21
6,APURIMAC,20
7,LAMBAYEQUE,20
8,CAJAMARCA,19
9,HUANUCO,18


## 10. Top 10 Provincias con mayor cantidad de distritos

In [14]:
query_10 = """
SELECT Nombre_Departamento, Nombre_Provincia, COUNT(DISTINCT Cod_Distrito) as Total_Distritos
FROM dim_geografia
GROUP BY Nombre_Departamento, Nombre_Provincia
ORDER BY Total_Distritos DESC
LIMIT 10
"""
spark.sql(query_10).toPandas()

,Nombre_Departamento,Nombre_Provincia,Total_Distritos
0,LIMA,LIMA,43
1,JUNIN,JAUJA,34
2,LIMA,YAUYOS,33
3,LIMA,HUAROCHIRI,32
4,AREQUIPA,AREQUIPA,29
5,JUNIN,HUANCAYO,28
6,AMAZONAS,LUYA,23
7,HUANCAVELICA,TAYACAJA,23
8,AMAZONAS,CHACHAPOYAS,21
9,AYACUCHO,LUCANAS,21


## 11. Presupuesto PIM Total a nivel nacional por año

In [15]:
query_11 = """
SELECT ANO_DOC as Anio, ROUND(SUM(Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec
GROUP BY ANO_DOC
ORDER BY ANO_DOC DESC
"""
spark.sql(query_11).toPandas()

,Anio,PIM_Millones
0,2026,41237.32
1,2025,43231.18
2,2024,42050.60
3,2023,40030.48
4,2022,46232.51
5,2021,41981.16
6,2020,34944.26
7,2019,30619.26
8,2018,33802.15
9,2017,29127.77


## 12. PIA vs PIM Total a nivel nacional por año

In [16]:
query_12 = """
SELECT ANO_DOC as Anio, 
       ROUND(SUM(Monto_PIA) / 1000000, 2) as PIA_Millones,
       ROUND(SUM(Monto_PIM) / 1000000, 2) as PIM_Millones,
       ROUND((SUM(Monto_PIM) - SUM(Monto_PIA)) / 1000000, 2) as Diferencia_Millones
FROM fact_ejec
GROUP BY ANO_DOC
ORDER BY ANO_DOC DESC
"""
spark.sql(query_12).toPandas()

,Anio,PIA_Millones,PIM_Millones,Diferencia_Millones
0,2026,30394.73,41237.32,10842.58
1,2025,28435.13,43231.18,14796.05
2,2024,29706.50,42050.60,12344.09
3,2023,25117.22,40030.48,14913.26
4,2022,19281.18,46232.51,26951.33
5,2021,19612.83,41981.16,22368.33
6,2020,17975.45,34944.26,16968.81
7,2019,17176.95,30619.26,13442.30
8,2018,15386.26,33802.15,18415.89
9,2017,14553.45,29127.77,14574.32


## 13. Top 10 municipalidades con mayor PIM histórico acumulado

In [17]:
query_13 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Total_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Ejecutora
ORDER BY PIM_Total_Millones DESC
LIMIT 10
"""
spark.sql(query_13).toPandas()

,Ejecutora,PIM_Total_Millones
0,MUNICIPALIDAD METROPOLITANA DE LIMA,32816.92
1,MUNICIPALIDAD DISTRITAL DE SAN MARCOS,8136.62
2,MUNICIPALIDAD PROVINCIAL DEL CALLAO,6186.92
3,MUNICIPALIDAD DISTRITAL DE ECHARATI,4840.21
4,MUNICIPALIDAD DISTRITAL DE SANTIAGO DE SURCO,4621.14
5,MUNICIPALIDAD DISTRITAL DE MIRAFLORES,4350.85
6,MUNICIPALIDAD DISTRITAL DE SAN ISIDRO,4059.59
7,MUNICIPALIDAD PROVINCIAL DEL SANTA - CHIMBOTE,3966.46
8,MUNICIPALIDAD PROVINCIAL DE TRUJILLO,3892.07
9,MUNICIPALIDAD DISTRITAL DE MEGANTONI,3687.30


## 14. Top 10 municipalidades con mayor Monto Recaudado histórico acumulado

In [18]:
query_14 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Ejecutora
ORDER BY Recaudado_Millones DESC
LIMIT 10
"""
spark.sql(query_14).toPandas()

,Ejecutora,Recaudado_Millones
0,MUNICIPALIDAD METROPOLITANA DE LIMA,44738.99
1,MUNICIPALIDAD DISTRITAL DE SAN MARCOS,8076.73
2,MUNICIPALIDAD PROVINCIAL DEL CALLAO,5224.50
3,MUNICIPALIDAD DISTRITAL DE MIRAFLORES,4555.53
4,MUNICIPALIDAD DISTRITAL DE ECHARATI,4504.08
5,MUNICIPALIDAD DISTRITAL DE SANTIAGO DE SURCO,4445.80
6,MUNICIPALIDAD DISTRITAL DE SAN ISIDRO,3931.27
7,MUNICIPALIDAD PROVINCIAL DEL SANTA - CHIMBOTE,3883.88
8,MUNICIPALIDAD PROVINCIAL DE TRUJILLO,3856.76
9,MUNICIPALIDAD PROVINCIAL DE AREQUIPA,3855.16


## 15. Promedio de Tasa de Avance (Recaudado/PIM) por departamento en el último año

In [19]:
query_15 = """
SELECT g.Nombre_Departamento, 
       ROUND(AVG(f.Tasa_Avance), 4) as Tasa_Avance_Promedio
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE f.ANO_DOC = (SELECT MAX(ANO_DOC) FROM fact_ejec)
GROUP BY g.Nombre_Departamento
ORDER BY Tasa_Avance_Promedio DESC
"""
spark.sql(query_15).toPandas()

,Nombre_Departamento,Tasa_Avance_Promedio
0,ICA,0.2058
1,AREQUIPA,0.2015
2,TUMBES,0.1308
3,LAMBAYEQUE,0.1263
4,ANCASH,0.1240
5,PIURA,0.1170
6,LIMA,0.0882
7,LA LIBERTAD,0.0853
8,MOQUEGUA,0.0652
9,TACNA,0.0616


## 16. Monto PIM por Fuente de Financiamiento (Top 10 histórico)

In [20]:
query_16 = """
SELECT fi.Fuente_Financiamiento, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
GROUP BY fi.Fuente_Financiamiento
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_16).toPandas()

,Fuente_Financiamiento,PIM_Millones
0,RECURSOS DETERMINADOS,368153.28
1,RECURSOS DIRECTAMENTE RECAUDADOS,63749.61
2,RECURSOS POR OPERACIONES OFICIALES DE CREDITO,59500.95
3,DONACIONES Y TRANSFERENCIAS,25597.17


## 17. Monto PIM por Rubro de Financiamiento (Top 10 histórico)

In [21]:
query_17 = """
SELECT fi.Rubro, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
GROUP BY fi.Rubro
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_17).toPandas()

,Rubro,PIM_Millones
0,"CANON Y SOBRECANON, REGALIAS, RENTA DE ADUANAS...",195459.15
1,FONDO DE COMPENSACION MUNICIPAL,114411.67
2,RECURSOS DIRECTAMENTE RECAUDADOS,63749.61
3,RECURSOS POR OPERACIONES OFICIALES DE CREDITO,59500.95
4,IMPUESTOS MUNICIPALES,58282.46
5,DONACIONES Y TRANSFERENCIAS,25597.17


## 18. PIM por Genérica de Ingreso (Top 10)

In [22]:
query_18 = """
SELECT c.Generica, ROUND(SUM(f.Monto_PIM) / 1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_clasificacion c ON f.SK_Clasificacion = c.SK_Clasificacion
GROUP BY c.Generica
ORDER BY PIM_Millones DESC
LIMIT 10
"""
spark.sql(query_18).toPandas()

,Generica,PIM_Millones
0,DONACIONES Y TRANSFERENCIAS,245236.42
1,SALDOS DE BALANCE,117097.78
2,IMPUESTOS Y CONTRIBUCIONES OBLIGATORIAS,52578.66
3,ENDEUDAMIENTO,44899.22
4,VENTA DE BIENES Y SERVICIOS Y DERECHOS ADMINIS...,37646.97
5,OTROS INGRESOS,19005.03
6,VENTA DE ACTIVOS NO FINANCIEROS,533.66
7,CONTRIBUCIONES SOCIALES,3.28
8,VENTA DE ACTIVOS FINANCIEROS,0.00


## 19. Ejecución Presupuestal (Recaudado) por Semestre en los últimos 3 años

In [23]:
query_19 = """
SELECT f.ANO_DOC, t.Semestre, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
WHERE f.ANO_DOC >= (SELECT MAX(ANO_DOC) - 2 FROM fact_ejec)
GROUP BY f.ANO_DOC, t.Semestre
ORDER BY f.ANO_DOC DESC, t.Semestre DESC
"""
spark.sql(query_19).toPandas()

,ANO_DOC,Semestre,Recaudado_Millones
0,2026,1,26103.33
1,2025,2,13786.04
2,2025,1,32385.80
3,2024,2,17097.31
4,2024,1,27851.18


## 20. Total Recaudado por Mes en el último año disponible

In [24]:
query_20 = """
SELECT t.Mes, t.Nombre_Mes, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
WHERE f.ANO_DOC = (SELECT MAX(ANO_DOC) FROM fact_ejec)
GROUP BY t.Mes, t.Nombre_Mes
ORDER BY t.Mes
"""
spark.sql(query_20).toPandas()

,Mes,Nombre_Mes,Recaudado_Millones
0,1,ENERO,9536.43
1,2,FEBRERO,7319.61
2,3,MARZO,3558.38
3,4,ABRIL,2254.86
4,5,MAYO,2757.86
5,6,JUNIO,676.20


## 21. Top 10 Departamentos con mayor Brecha Presupuestal histórica

In [25]:
query_21 = """
SELECT g.Nombre_Departamento, ROUND(SUM(f.Brecha_Presupuestal) / 1000000, 2) as Brecha_Total_Millones
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
GROUP BY g.Nombre_Departamento
ORDER BY Brecha_Total_Millones DESC
LIMIT 10
"""
spark.sql(query_21).toPandas()

,Nombre_Departamento,Brecha_Total_Millones
0,PIURA,3752.32
1,LORETO,2225.86
2,CUSCO,2165.49
3,LA LIBERTAD,1993.30
4,ICA,1848.29
5,CAJAMARCA,1583.34
6,PROVINCIA CONSTITUCIONAL DEL CALLAO,1574.60
7,AYACUCHO,1218.45
8,PUNO,1186.57
9,LAMBAYEQUE,1142.70


## 22. Cantidad de registros por Nivel de Calidad (SK_Calidad)

In [26]:
query_22 = """
SELECT SK_Calidad, 
       CASE 
         WHEN SK_Calidad = 1 THEN 'Limpio'
         WHEN SK_Calidad = 2 THEN 'Advertencias menores'
         WHEN SK_Calidad = 3 THEN 'Inconsistencias catálogo/jerarquía'
         WHEN SK_Calidad = 4 THEN 'Error crítico'
         ELSE 'Desconocido'
       END as Descripcion_Calidad,
       COUNT(*) as Total_Registros
FROM fact_ejec
GROUP BY SK_Calidad
ORDER BY SK_Calidad
"""
spark.sql(query_22).toPandas()

,SK_Calidad,Descripcion_Calidad,Total_Registros
0,1,Limpio,8896130
1,2,Advertencias menores,89545
2,4,Error crítico,13548


## 23. Top 10 municipalidades con más Errores Críticos (SK_Calidad = 4)

In [27]:
query_23 = """
SELECT m.Ejecutora, COUNT(*) as Registros_Criticos
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
WHERE f.SK_Calidad = 4
GROUP BY m.Ejecutora
ORDER BY Registros_Criticos DESC
LIMIT 10
"""
spark.sql(query_23).toPandas()

,Ejecutora,Registros_Criticos
0,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,82
1,MUNICIPALIDAD DISTRITAL DE INDEPENDENCIA,79
2,MUNICIPALIDAD DISTRITAL DE PICHARI,68
3,MUNICIPALIDAD DISTRITAL DE SAN ANTONIO,67
4,MUNICIPALIDAD DISTRITAL DE PUEBLO NUEVO,57
5,MUNICIPALIDAD PROVINCIAL DE LA CONVENCION - SA...,54
6,MUNICIPALIDAD DISTRITAL DE SAN JERONIMO,53
7,MUNICIPALIDAD PROVINCIAL DE CAYLLOMA - CHIVAY,48
8,MUNICIPALIDAD PROVINCIAL DE PASCO - CHAUPIMARCA,48
9,MUNICIPALIDAD PROVINCIAL DE OXAPAMPA,47


## 24. Total recaudado en distritos que empiecen con "SAN" o "SANTA"

In [28]:
query_24 = """
SELECT g.Nombre_Distrito, ROUND(SUM(f.Monto_Recaudado) / 1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
WHERE g.Nombre_Distrito LIKE 'SAN %' OR g.Nombre_Distrito LIKE 'SANTA %'
GROUP BY g.Nombre_Distrito
ORDER BY Recaudado_Millones DESC
LIMIT 15
"""
spark.sql(query_24).toPandas()

,Nombre_Distrito,Recaudado_Millones
0,SAN MARCOS,8118.98
1,SAN ISIDRO,3969.19
2,SAN JUAN DE LURIGANCHO,3132.18
3,SAN MIGUEL,2476.78
4,SAN MARTIN DE PORRES,1965.87
5,SAN BORJA,1869.82
6,SANTA ANA,1770.78
7,SAN ANTONIO,1680.85
8,SAN SEBASTIAN,1317.75
9,SAN JUAN BAUTISTA,1222.10


## 25. Promedio de Monto_PIA por Categoría Municipal

In [29]:
query_25 = """
SELECT m.Categoria_Municipal, ROUND(AVG(f.Monto_PIA), 2) as Promedio_PIA
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
GROUP BY m.Categoria_Municipal
ORDER BY Promedio_PIA DESC
"""
spark.sql(query_25).toPandas()

,Categoria_Municipal,Promedio_PIA
0,C,102858.58
1,A,91571.73
2,D,36992.67
3,G,26375.86
4,B,24677.08
5,SIN_CATEGORIA,17830.01
6,F,14664.53
7,E,13386.12


## 26. PIM Promedio por Trimestre a nivel nacional

In [30]:
query_26 = """
SELECT t.Trimestre, ROUND(AVG(f.Monto_PIM), 2) as Promedio_PIM
FROM fact_ejec f
JOIN dim_tiempo t ON f.SK_Tiempo = t.SK_Tiempo
GROUP BY t.Trimestre
ORDER BY t.Trimestre
"""
spark.sql(query_26).toPandas()

,Trimestre,Promedio_PIM
0,1,130989.67
1,2,23502.78
2,3,21584.68
3,4,18291.75


## 27. Municipalidades con Tasa de Avance superior al 100% (posibles excesos de recaudación sobre PIM)

In [31]:
query_27 = """
SELECT m.Ejecutora, f.ANO_DOC, f.Monto_PIM, f.Monto_Recaudado, f.Tasa_Avance
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
WHERE f.Tasa_Avance > 1.0 AND f.Monto_PIM > 0
ORDER BY f.Tasa_Avance DESC
LIMIT 10
"""
spark.sql(query_27).toPandas()

,Ejecutora,ANO_DOC,Monto_PIM,Monto_Recaudado,Tasa_Avance
0,MUNICIPALIDAD DISTRITAL DE SAN MIGUEL,2014,1.00,176936.16,176936.160000
1,MUNICIPALIDAD DISTRITAL DE LONYA CHICO,2014,1.00,119800.92,119800.920000
2,MUNICIPALIDAD DISTRITAL DE SAN ANTONIO,2016,1.00,81894.20,81894.200000
3,MUNICIPALIDAD DISTRITAL DE LIVITACA,2013,1.00,40000.00,40000.000000
4,MUNICIPALIDAD DISTRITAL DE HUACAÃA,2017,1.00,36696.28,36696.280000
5,MUNICIPALIDAD DISTRITAL DE VISTA ALEGRE,2018,1.00,30189.00,30189.000000
6,MUNICIPALIDAD PROVINCIAL DEL CALLAO,2022,62.00,1347073.63,21726.994032
7,MUNICIPALIDAD DISTRITAL DE CACHACHI,2016,2.00,40002.00,20001.000000
8,MUNICIPALIDAD PROVINCIAL DE PADRE ABAD - AGUAITIA,2018,5.00,66154.89,13230.978000
9,MUNICIPALIDAD DISTRITAL DE HUAÃEC,2017,1.00,10000.00,10000.000000


## 28. Departamentos con más municipalidades en el país

In [32]:
query_28 = """
SELECT g.Nombre_Departamento, COUNT(DISTINCT m.SK_Municipalidad) as Total_Municipalidades
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
JOIN dim_geografia g ON f.SK_Geografia = g.SK_Geografia
GROUP BY g.Nombre_Departamento
ORDER BY Total_Municipalidades DESC
"""
spark.sql(query_28).toPandas()

,Nombre_Departamento,Total_Municipalidades
0,LIMA,178
1,ANCASH,170
2,JUNIN,133
3,CAJAMARCA,132
4,AYACUCHO,132
5,CUSCO,124
6,AREQUIPA,113
7,PUNO,112
8,HUANCAVELICA,107
9,AMAZONAS,88


## 29. Evolución del Rubro de Financiamiento "CANON Y SOBRECANON" por año

In [33]:
query_29 = """
SELECT f.ANO_DOC, ROUND(SUM(f.Monto_PIM)/1000000, 2) as PIM_Millones
FROM fact_ejec f
JOIN dim_financiamiento fi ON f.SK_Financiamiento = fi.SK_Financiamiento
WHERE fi.Rubro LIKE '%CANON%'
GROUP BY f.ANO_DOC
ORDER BY f.ANO_DOC DESC
"""
spark.sql(query_29).toPandas()

,ANO_DOC,PIM_Millones
0,2026,15176.08
1,2025,17218.20
2,2024,17063.65
3,2023,17815.47
4,2022,18794.39
5,2021,11419.57
6,2020,9643.47
7,2019,9499.16
8,2018,11290.51
9,2017,8746.95


## 30. Top 5 municipalidades con mayor recaudación de la Genérica "IMPUESTOS Y CONTRIBUCIONES OBLIGATORIAS"

In [34]:
query_30 = """
SELECT m.Ejecutora, ROUND(SUM(f.Monto_Recaudado)/1000000, 2) as Recaudado_Millones
FROM fact_ejec f
JOIN dim_municipalidad m ON f.SK_Municipalidad = m.SK_Municipalidad
JOIN dim_clasificacion c ON f.SK_Clasificacion = c.SK_Clasificacion
WHERE c.Generica LIKE '%IMPUESTOS%'
GROUP BY m.Ejecutora
ORDER BY Recaudado_Millones DESC
LIMIT 5
"""
spark.sql(query_30).toPandas()

,Ejecutora,Recaudado_Millones
0,MUNICIPALIDAD METROPOLITANA DE LIMA,9914.97
1,MUNICIPALIDAD DISTRITAL DE SANTIAGO DE SURCO,2129.73
2,MUNICIPALIDAD DISTRITAL DE MIRAFLORES,1851.78
3,MUNICIPALIDAD DISTRITAL DE SAN ISIDRO,1834.84
4,MUNICIPALIDAD PROVINCIAL DEL CALLAO,1161.21
